# 01 · Frequentist analysis of the raw asymmetry $A_{\rm raw}$ in $B^\pm\to K^\pm K^+K^-$

**What this notebook is.** A guided tour of *how* the frequentist part of the study was done, step by step: from the
raw candidates of the LHCb Open Data to the value of $A_{\rm raw}$ with its uncertainty, its significance and its
systematic uncertainty. It follows the order of the report (`short`) and, in the last sections, the checks of its
appendix (`cut`).

**What it is not.** It is not the production run of the analysis. To keep the whole notebook under ~30 minutes the
expensive parts (bootstrap, toy experiments, scans) use *fewer* replicas and *coarser* grids than in the report, so
the numbers can differ slightly from it. Next to every result you find the value quoted in the report
(**report:**) to compare the two; the method is identical.

**The observable.**
$$A_{\rm raw}=\frac{N(B^-)-N(B^+)}{N(B^-)+N(B^+)}$$
measured on the signal yields of $B^\pm\to K^\pm K^+K^-$ decays. It is a *raw* asymmetry: it is not corrected for
production and detection asymmetries, so it is not a measurement of $A_{CP}$.

### Map of the notebook

| § | Content | Report (short → cut appendix) |
|---|---|---|
| 1 | Data, preselection, mass | *Data*; App. *Dataset and preselection* |
| 2 | Mass model and likelihood | *Mass model and likelihood* |
| 3 | Fit: separate and simultaneous, Minos intervals, goodness of fit | *Frequentist analysis* |
| 4 | Profile likelihood, Wilks test of $A=0$ | *Frequentist analysis* |
| 5 | Coverage with a bootstrap of the data | *Frequentist analysis* |
| 6 | Toy experiments: bias, pulls, coverage, size and power, Neyman belt | *Validation of the estimator* |
| 7 | Systematic uncertainty: 13 variants and the combination rule | *Result and systematics* |
| 8 | Stability scans and fine scan of the PID cut | App. *Systematics* |
| 9 | Local asymmetry in the Dalitz plane, charm vetoes, region of the published analysis | *Dalitz plane* |
| 10 | Summary | *Introduction*, Table 1 |

Parts that go beyond the techniques of the course are tagged **[EXTRA]**.

## 0 · Setup

In [ ]:
import os, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
from scipy.stats import norm, chi2
from iminuit import Minuit, cost

import plot_style
plot_style.use()
C = plot_style.COLORS
T0 = time.time()

# Data: put the three .root files of CERN Open Data record 4900 in ./data (or edit the path)
DATA_DIR = "data" if os.path.isdir("data") else "../../opendata-project/Data"
np.random.seed(1)

M_K = 493.677            # charged-kaon mass [MeV]
LO, HI = 5100.0, 5600.0  # nominal fit window of the mass [MeV]

## 1 · Data and preselection

The record contains $B^\pm\to h^\pm h^+h^-$ candidates (2011 data, 7 TeV), one file per magnet polarity. Every
candidate has the momentum, the particle-identification (PID) likelihoods and the charge of the three tracks
$H_1,H_2,H_3$. We read only the branches we need.

In [ ]:
tracks = (1, 2, 3)
branches = [f"H{i}_{v}" for i in tracks for v in ("PX", "PY", "PZ", "ProbK", "ProbPi", "isMuon", "Charge")]

frames = []
for pol in ("Down", "Up"):
    with uproot.open(f"{DATA_DIR}/B2HHH_Magnet{pol}.root") as f:
        d = f["DecayTree"].arrays(branches, library="pd")
    d["magnet"] = pol.lower()
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print("candidates in the record:", len(df))

**Preselection.** We want $B^\pm\to K^\pm K^+K^-$: all three tracks must look like kaons and not like pions or muons,

$$\texttt{ProbPi}<0.5\ \wedge\ \texttt{ProbK}>0.5\ \wedge\ \lnot\texttt{isMuon}\quad\text{for each of the three tracks.}$$

The cut flow below gives the effect of each requirement alone and cumulatively (cut appendix, Table 1).

In [ ]:
not_muon = np.ones(len(df), bool); pi_ok = np.ones(len(df), bool); k_ok = np.ones(len(df), bool)
for i in tracks:
    not_muon &= ~df[f"H{i}_isMuon"].to_numpy().astype(bool)
    pi_ok    &= df[f"H{i}_ProbPi"].to_numpy() < 0.5
    k_ok     &= df[f"H{i}_ProbK"].to_numpy() > 0.5

print(f"{'requirement':28s} {'alone':>10s} {'cumulative':>11s}")
cum = np.ones(len(df), bool)
for name, mask in (("not muon (3 tracks)", not_muon), ("ProbPi < 0.5 (3 tracks)", pi_ok), ("ProbK > 0.5 (3 tracks)", k_ok)):
    cum &= mask
    print(f"{name:28s} {mask.sum():10d} {cum.sum():11d}")
print("report: 6311517 / 975724 / 28812 alone, 23895 cumulative")

**Invariant mass.** We give the kaon mass to the three tracks, sum the four-momenta and take $m^2=E^2-|\vec p|^2$.
The two opposite-sign pairs $m(H_1H_2)$ and $m(H_1H_3)$ are kept for the Dalitz plane ($H_1$ is always the kaon with
charge opposite to the $B$).

We keep a slightly looser sample (`ProbK > 0.3`, `ProbPi < 0.7`) and store the *weakest* PID value of each
candidate: later the systematic study and the PID scan simply re-apply tighter cuts on these columns.

In [ ]:
loose = np.ones(len(df), bool)
for i in tracks:
    loose &= (df[f"H{i}_ProbK"] > 0.3) & (df[f"H{i}_ProbPi"] < 0.7) & ~df[f"H{i}_isMuon"].astype(bool)
d = df[loose].reset_index(drop=True)
del df

def invariant_mass(d, particles):
    """Invariant mass [MeV] of the listed tracks, kaon mass hypothesis."""
    E = sum(np.sqrt(d[f"H{i}_PX"]**2 + d[f"H{i}_PY"]**2 + d[f"H{i}_PZ"]**2 + M_K**2) for i in particles)
    px, py, pz = (sum(d[f"H{i}_{c}"] for i in particles) for c in ("PX", "PY", "PZ"))
    return np.sqrt(np.clip(E**2 - px**2 - py**2 - pz**2, 0, None))

d["m"]  = invariant_mass(d, (1, 2, 3))
d["m12"], d["m13"] = invariant_mass(d, (1, 2)), invariant_mass(d, (1, 3))
d["q"]  = d.H1_Charge + d.H2_Charge + d.H3_Charge              # charge of the B
d["pT"] = np.sqrt(sum(d[f"H{i}_PX"] for i in tracks)**2 + sum(d[f"H{i}_PY"] for i in tracks)**2)
d["p"]  = np.sqrt(d.pT**2 + sum(d[f"H{i}_PZ"] for i in tracks)**2)
d["min_probk"]  = d[[f"H{i}_ProbK" for i in tracks]].min(axis=1)
d["max_probpi"] = d[[f"H{i}_ProbPi" for i in tracks]].max(axis=1)
d = d[np.abs(d.q) == 1]
assert (d.H1_Charge == -d.q).all()          # H1 has the opposite charge of the B

nominal = d[(d.min_probk > 0.5) & (d.max_probpi < 0.5)]
sample = nominal[(nominal.m > LO) & (nominal.m < HI)].reset_index(drop=True)
print(f"preselected: {len(nominal)} (report 23895);  in the window [{LO:.0f},{HI:.0f}]: {len(sample)} (report 19731)")
m_pos = sample.m[sample.q > 0].to_numpy()       # B+
m_neg = sample.m[sample.q < 0].to_numpy()       # B-
n_pos, n_neg = len(m_pos), len(m_neg)

**A first number: counting.** Without any fit, the asymmetry of *all* candidates (signal and background together) is
$A_{\rm count}=(N^--N^+)/(N^-+N^+)$ with binomial uncertainty $\sqrt{(1-A^2)/N}$.

In [ ]:
def count_asym(q):
    n = len(q); a = (np.sum(q < 0) - np.sum(q > 0)) / n
    return a, np.sqrt((1 - a**2) / n)

a_c, da_c = count_asym(sample.q.to_numpy())
print(f"N(B+) = {n_pos}, N(B-) = {n_neg};  A_count = {a_c:+.4f} +- {da_c:.4f}   (report: -0.0432 +- 0.0071)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.arange(LO, HI + 1, 5)
ax[0].hist(m_pos, bins, histtype="step", color=C["pos"], label="$B^+$")
ax[0].hist(m_neg, bins, histtype="step", color=C["neg"], label="$B^-$")
ax[0].set(yscale="log", xlabel="m(KKK) [MeV]", ylabel="candidates / 5 MeV"); ax[0].legend()
for pol in ("down", "up"):
    s = sample[sample.magnet == pol]
    ax[1].hist(s.m, bins, histtype="step", color=C[pol], label=f"magnet {pol} ({len(s)})")
ax[1].set(yscale="log", xlabel="m(KKK) [MeV]"); ax[1].legend()
plt.tight_layout(); plt.show()

The peak at $m\approx5284$ MeV over a falling background has the same shape for the two charges and the two
polarities: this allows a mass $\mu$ and a resolution $\sigma$ *common* to $B^+$ and $B^-$ in the model.

## 2 · Mass model and likelihood

For each charge the mass spectrum is a mixture of signal and background on the window $[5100,5600]$ MeV,

$$f^\pm(m)=N_s^\pm\,S(m;\mu,\sigma)+N_b^\pm\,B^\pm(m),\qquad
N_s^\pm=\tfrac{N_s}{2}(1\mp A),$$

* **signal** $S$: Gaussian, truncated and normalised to the window; $\mu,\sigma$ are *shared* by $B^+$ and $B^-$;
* **background** $B^\pm$: sum of two exponentials, $f\,E(\lambda_1)+(1-f)\,E(\lambda_2)$, with a shape and a yield of
  its own for each charge (the steep exponential describes the rise of the background towards the lower edge);
* **$A$** is a parameter of the fit: it is the *parameter of interest* (POI); the other 11 are *nuisance parameters*.

The likelihood is *extended* (the yields are parameters, the number of candidates is Poisson):

$$-\ln\mathcal L=(N_s+N_b)-\sum_{\pm}\sum_{i\in\pm}\ln\!\big[N_s^\pm S(m_i)+N_b^\pm B^\pm(m_i)\big]+\text{const.}$$

We write the pdfs ourselves (a few lines each) and let `iminuit` build the extended likelihood with
`cost.ExtendedUnbinnedNLL`: the model is a function `model(m, *parameters)` that returns the expected total number
of events and the density of events at $m$.

In [ ]:
def gauss_pdf(m, mu, sigma, lo=LO, hi=HI):
    """Gaussian normalised on [lo, hi]."""
    return norm.pdf(m, mu, sigma) / (norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma))

def exp_pdf(m, lam, lo=LO, hi=HI):
    """Decreasing exponential exp(-lam (m - lo)) normalised on [lo, hi]."""
    return lam * np.exp(-lam * (m - lo)) / (1 - np.exp(-lam * (hi - lo)))

def bkg_pdf(m, lam1, lam2, f, lo=LO, hi=HI):
    """Background: two exponentials, weight f on the steep one (lam1 > lam2)."""
    return f * exp_pdf(m, lam1, lo, hi) + (1 - f) * exp_pdf(m, lam2, lo, hi)

# one model per charge; the parameters with the same name (N_s, A, mu, sigma) are shared in the joint fit
def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
    n_sig = N_s * (1 - A) / 2                       # B+ : (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)

def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2                       # B- : (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

## 3 · The fit

### 3.1 Separate fits (a cross-check)

First each charge is fitted alone (seven free parameters: yields, $\mu$, $\sigma$, background shape) and the
asymmetry is computed from the two signal yields, $A=(N_s^--N_s^+)/(N_s^-+N_s^+)$, propagating the uncertainties.

In [ ]:
def model_single(m, N_s, mu, sigma, N_b, lam1, lam2, f):
    return N_s + N_b, N_s * gauss_pdf(m, mu, sigma) + N_b * bkg_pdf(m, lam1, lam2, f)

def fit_single(m_data):
    nll = cost.ExtendedUnbinnedNLL(m_data, model_single)
    fit = Minuit(nll, N_s=0.78 * len(m_data), mu=5284.0, sigma=18.0, N_b=0.22 * len(m_data), lam1=0.03, lam2=0.002, f=0.15)
    fit.limits["mu"] = (5260, 5310); fit.limits["sigma"] = (3, 40)
    fit.limits["lam1"] = (1e-5, 0.1); fit.limits["lam2"] = (1e-5, 0.1); fit.limits["f"] = (0, 1)
    fit.limits["N_s"] = (0, 2 * len(m_data)); fit.limits["N_b"] = (0, 2 * len(m_data))
    fit.migrad(); fit.hesse()
    return fit

fit_p, fit_n = fit_single(m_pos), fit_single(m_neg)
Np, dNp = fit_p.values["N_s"], fit_p.errors["N_s"]
Nn, dNn = fit_n.values["N_s"], fit_n.errors["N_s"]
A_sep = (Nn - Np) / (Nn + Np)
dA_sep = 2 * np.sqrt((Np * dNn)**2 + (Nn * dNp)**2) / (Nn + Np)**2
print(f"valid: {fit_p.valid}, {fit_n.valid};  N_s(B+) = {Np:.0f} +- {dNp:.0f},  N_s(B-) = {Nn:.0f} +- {dNn:.0f}")
print(f"A_raw (separate fits) = {A_sep:+.4f} +- {dA_sep:.4f}     (report: -0.0442 +- 0.0087)")

### 3.2 Simultaneous fit: the nominal result

The two charges are fitted *together*: the total cost is the sum of the two extended likelihoods and $A$ is a
parameter of it, so the profile likelihood of $A$ is available directly. `Migrad` minimises, `Hesse` gives the
symmetric uncertainties from the curvature (inverse of the Fisher information) and `Minos` the asymmetric
intervals from the profile likelihood.

In [ ]:
def fit_joint(m_p, m_n, start, a_fixed=None):
    """Simultaneous fit of B+ and B-: the cost is the sum of the two extended likelihoods.
    With a_fixed the asymmetry is held at that value (profile likelihood at that point)."""
    nll = cost.ExtendedUnbinnedNLL(m_p, model_pos) + cost.ExtendedUnbinnedNLL(m_n, model_neg)
    f = Minuit(nll, **start)
    f.limits["A"] = (-1, 1); f.limits["mu"] = (5260, 5310); f.limits["sigma"] = (3, 40)
    f.limits["N_s"] = (0, 2 * (len(m_p) + len(m_n)))
    for s in ("pos", "neg"):
        f.limits[f"N_b_{s}"] = (0, 2 * (len(m_p) + len(m_n)))
        f.limits[f"lam1_{s}"] = (1e-5, 0.1); f.limits[f"lam2_{s}"] = (1e-5, 0.1); f.limits[f"f_{s}"] = (0, 1)
    if a_fixed is not None:
        f.values["A"] = a_fixed
        f.fixed["A"] = True
    f.migrad()
    return f

start = dict(N_s=0.78 * len(sample), A=-0.04, mu=5284.0, sigma=18.0,
             N_b_pos=0.22 * n_pos, lam1_pos=0.03, lam2_pos=0.002, f_pos=0.15,
             N_b_neg=0.22 * n_neg, lam1_neg=0.03, lam2_neg=0.002, f_neg=0.15)
fit = fit_joint(m_pos, m_neg, start)
fit.migrad(); fit.hesse()
fit.minos("A")                                      # 68 % interval (default cl = 1 sigma)
ci68 = (fit.values["A"] + fit.merrors["A"].lower, fit.values["A"] + fit.merrors["A"].upper)
fit_95 = copy.deepcopy(fit); fit_95.minos("A", cl=0.95)
ci95 = (fit_95.values["A"] + fit_95.merrors["A"].lower, fit_95.values["A"] + fit_95.merrors["A"].upper)

assert fit.valid and fit.values["lam1_pos"] > fit.values["lam2_pos"] and fit.values["lam1_neg"] > fit.values["lam2_neg"]
for k in ("N_s", "A", "mu", "sigma"):
    print(f"{k:8s} = {fit.values[k]:12.4f} +- {fit.errors[k]:.4f}")
print(f"N_b = {fit.values['N_b_pos'] + fit.values['N_b_neg']:.0f}")
print(f"\nA_raw = {fit.values['A']:+.4f} +- {fit.errors['A']:.4f} (Hesse)")
print(f"  68% Minos [{ci68[0]:+.4f}, {ci68[1]:+.4f}]    report: [-0.0526, -0.0356]")
print(f"  95% Minos [{ci95[0]:+.4f}, {ci95[1]:+.4f}]    report: [-0.0607, -0.0275]")
print("report: A = -0.0441 +- 0.0085, N_s = 15328 +- 133")

### 3.3 Goodness of fit and picture

The $\chi^2=\sum_i(n_i-\nu_i)^2/\nu_i$ in bins of 10 MeV (both charges, $\nu_i$ from the fitted model) with
$\mathrm{ndof}=2\times50-12$. The pulls $(n_i-\nu_i)/\sqrt{\nu_i}$ show *where* the model fails. (The model is not
perfect: the signal tails are heavier than a Gaussian; $A_{\rm raw}$ does not depend on it, see §7.)

In [ ]:
def expected_counts(model, args, edges, n_sub=5):
    """Expected counts per bin: mean of the density at n_sub points in each bin, times the width."""
    width = edges[1] - edges[0]
    offs = (np.arange(n_sub) + 0.5) / n_sub
    pts = edges[:-1, None] + offs[None, :] * width
    return model(pts, *args)[1].mean(axis=1) * width

edges = np.arange(LO, HI + 1e-9, 10.0)
centers = 0.5 * (edges[1:] + edges[:-1])
v = fit.values
nu_pos = expected_counts(model_pos, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_pos", "lam1_pos", "lam2_pos", "f_pos")], edges)
nu_neg = expected_counts(model_neg, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_neg", "lam1_neg", "lam2_neg", "f_neg")], edges)
obs_pos, obs_neg = np.histogram(m_pos, edges)[0], np.histogram(m_neg, edges)[0]
chi2_val = np.sum((obs_pos - nu_pos)**2 / nu_pos) + np.sum((obs_neg - nu_neg)**2 / nu_neg)
ndof = 2 * len(centers) - 12
print(f"chi2/ndof = {chi2_val:.1f}/{ndof},  p = {chi2.sf(chi2_val, ndof):.1e}     (report: 160/88, p = 4.8e-6)")

fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for col, (obs, nu, lab, c) in enumerate(((obs_pos, nu_pos, "$B^+$", C["pos"]), (obs_neg, nu_neg, "$B^-$", C["neg"]))):
    ax[0, col].errorbar(centers, obs, np.sqrt(obs), fmt="o", ms=3, color="k", label="data")
    ax[0, col].plot(centers, nu, color=C["model"], label="fit")
    ax[0, col].set(yscale="log", ylabel="candidates / 10 MeV", title=lab); ax[0, col].legend()
    ax[1, col].bar(centers, (obs - nu) / np.sqrt(nu), width=9, color=c)
    ax[1, col].set(xlabel="m(KKK) [MeV]", ylabel="pull")
plt.tight_layout(); plt.show()

## 4 · Profile likelihood and test of $A=0$

The **profile likelihood ratio** re-minimises the 11 nuisance parameters at each value of $A$,
$$q(A)=-2\ln\frac{\mathcal L(A,\hat{\hat\theta}(A))}{\mathcal L(\hat A,\hat\theta)} .$$
By Wilks' theorem $q(A_{\rm true})\sim\chi^2_1$, so the 68 % and 95 % intervals are the sets $q\le1$ and $q\le3.84$
(the Minos intervals above). The null hypothesis $H_0:A=0$ is tested with $q(0)$: $p=P(\chi^2_1>q(0))$ and the
significance is $Z=\sqrt{q(0)}$.

In [ ]:
def q_at(A0, best=fit):
    """Profile likelihood ratio at A = A0: fix A, re-minimise everything else."""
    f0 = copy.deepcopy(best)
    f0.values["A"] = A0
    f0.fixed["A"] = True
    f0.migrad()
    return f0.fval - best.fval            # iminuit cost functions return -2 ln L already

q0 = q_at(0.0)
print(f"q(0) = {q0:.1f},  p = {chi2.sf(q0, 1):.2e},  Z = sqrt(q) = {np.sqrt(q0):.2f} sigma     (report: q ~ 27, p = 2.05e-7, 5.20 sigma)")

a_grid, q_grid, _ = fit.mnprofile("A", size=30, bound=(-0.085, 0.0))
q_grid = q_grid - fit.fval            # fval = -2 ln L
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(a_grid, q_grid, "o-", color=C["freq"], ms=3)
ax.axhline(1, ls="--", color="gray"); ax.axhline(3.84, ls=":", color="gray")
ax.set(xlabel="$A_{\\rm raw}$", ylabel="$q(A)$", ylim=(0, 30))
plt.tight_layout(); plt.show()

### 3.4 Counting with sideband subtraction (a model-light cross-check) [EXTRA]

A different, simpler estimate: fit the *background alone* on the sidebands (window without
$[5200,5350]$ MeV), extrapolate it under the peak to get the expected background $b$ in the signal region, and
subtract it from the observed number of candidates. For each charge $S=N_{\rm obs}-b$, then
$A=(S^--S^+)/(S^-+S^+)$.

In [ ]:
SR = (5200.0, 5350.0)                                        # signal region
grid = np.linspace(LO, HI, 5001)
in_sb = (grid < SR[0]) | (grid > SR[1])
in_sr = ~in_sb

def sideband_background(m_side):
    """Fit of the two-exponential background on the sidebands; returns the expected background in the signal region."""
    m_sb = m_side[(m_side < SR[0]) | (m_side > SR[1])]
    def pdf_sb(m, lam1, lam2, f):                           # background pdf renormalised on the sidebands
        norm_sb = np.trapezoid(bkg_pdf(grid, lam1, lam2, f)[in_sb], grid[in_sb])
        return bkg_pdf(m, lam1, lam2, f) / norm_sb
    f = Minuit(cost.UnbinnedNLL(m_sb, pdf_sb), lam1=0.03, lam2=0.002, f=0.15)
    f.limits["lam1"] = (1e-5, 0.1); f.limits["lam2"] = (1e-5, 0.1); f.limits["f"] = (0, 1)
    f.migrad()
    p = bkg_pdf(grid, *f.values)
    frac_sr = np.trapezoid(p[in_sr], grid[in_sr]) / np.trapezoid(p[in_sb], grid[in_sb])
    return len(m_sb) * frac_sr, len(m_sb)

S, dS = {}, {}
for lab, m_side in (("pos", m_pos), ("neg", m_neg)):
    b, n_sb = sideband_background(m_side)
    n_obs = np.sum((m_side > SR[0]) & (m_side < SR[1]))
    S[lab], dS[lab] = n_obs - b, np.sqrt(n_obs + b**2 / n_sb)
tot = S["pos"] + S["neg"]
a_sb = (S["neg"] - S["pos"]) / tot
da_sb = 2 * np.sqrt((S["pos"] * dS["neg"])**2 + (S["neg"] * dS["pos"])**2) / tot**2
print(f"A_raw (sideband subtraction) = {a_sb:+.4f} +- {da_sb:.4f}     (report: -0.0503 +- 0.0083)")

## 5 · Coverage of the Wilks interval: bootstrap of the data

Wilks' theorem is *asymptotic*. To check that the interval $q\le1$ really contains the true value 68 % of the time
we resample the data: each replica is a sample of $N$ candidates drawn **with replacement** from the observed ones
(`np.random.choice`, as in the course), refitted with the nominal model. We take $A_{\rm true}$ equal to the fitted
value; the 68 % interval of a replica contains it if and only if $t=q(A_{\rm true})\le 1$, where
$q(A_{\rm true})$ is evaluated *on the replica*.

*Report:* 1000 replicas, coverage $68.4\pm1.5\,\%$. Here: 100.

In [ ]:
A_hat = fit.values["A"]
best_vals = {k: fit.values[k] for k in start}                # start every replica from the best fit
best_vals["A"] = A_hat

def one_replica(m_p, m_n, a_true):
    """Free fit + fit with A fixed at a_true: returns (A_hat, sigma_hat, t = q(a_true), q(0), valid)."""
    f1 = fit_joint(m_p, m_n, best_vals); f1.hesse()
    f2 = fit_joint(m_p, m_n, best_vals, a_true)
    f3 = fit_joint(m_p, m_n, best_vals, 0.0)
    ok = f1.valid and f2.valid and f3.valid
    return f1.values["A"], f1.errors["A"], f2.fval - f1.fval, f3.fval - f1.fval, ok

N_BOOT = 100
t0 = time.time()
rows = []
q_all = sample.q.to_numpy()
m_all = sample.m.to_numpy()
for b in range(N_BOOT):
    idx = np.random.choice(len(sample), len(sample), replace=True)     # resample candidates with replacement
    rows.append(one_replica(m_all[idx][q_all[idx] > 0], m_all[idx][q_all[idx] < 0], A_hat))
boot = pd.DataFrame(rows, columns=["A", "sigma", "t", "q0", "valid"])
boot = boot[boot.valid]
cov68 = np.mean(boot.t <= 1.0); err68 = np.sqrt(cov68 * (1 - cov68) / len(boot))
pulls = (boot.A - A_hat) / boot.sigma
print(f"{len(boot)} valid replicas ({time.time() - t0:.0f} s)")
print(f"coverage of the 68.3% interval: {cov68:.3f} +- {err68:.3f}     (report: 0.684 +- 0.015)")
print(f"pulls: mean {pulls.mean():+.2f}, width {pulls.std():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(boot.t, bins=np.linspace(0, 8, 33), density=True, color=C["freq"], alpha=0.6, label="replicas")
x = np.linspace(0.05, 8, 200); ax[0].plot(x, chi2.pdf(x, 1), "k--", label="$\\chi^2_1$")
ax[0].axvline(1, color="gray", ls=":"); ax[0].set(xlabel="$t=q(A_{true})$", ylim=(0, 1.5)); ax[0].legend()
ax[1].hist(pulls, bins=np.linspace(-4, 4, 33), density=True, color=C["freq"], alpha=0.6)
x = np.linspace(-4, 4, 200); ax[1].plot(x, norm.pdf(x), "k--"); ax[1].set(xlabel="pull $(\\hat A-A_{true})/\\sigma$")
plt.tight_layout(); plt.show()

## 6 · Validation with toy experiments

The bootstrap resamples the *data*; here we generate **pseudo-experiments from the fitted model**, where we *know*
the true value, and check the estimator and the test:

* **bias** of $\hat A$ and **pulls** $(\hat A-A_{\rm true})/\sigma(\hat A)$ (mean 0, width 1 if unbiased and at the
  Cramér–Rao bound);
* **coverage** of $q(A_{\rm true})\le1$ and $\le3.84$;
* **size** of the test of $A=0$ (fraction of rejections at $A_{\rm true}=0$, nominal 5 %) and **power** (fraction
  of rejections at $A_{\rm true}\neq0$);
* the **Neyman construction**: the central 68 %/95 % ranges of $\hat A$ as a function of $A_{\rm true}$ (the
  *confidence belt*), read at the observed $\hat A$.

The toy is built by the **transformation method** (inverse cdf of uniform random numbers) for the Gaussian
and for the exponentials, with Poisson yields.

In [ ]:
def sample_gauss(mu, sigma, n):
    u = np.random.uniform(norm.cdf(LO, mu, sigma), norm.cdf(HI, mu, sigma), n)
    return mu + sigma * norm.ppf(u)

def sample_exp(lam, n):
    u = np.random.uniform(size=n)
    return LO - np.log(1 - u * (1 - np.exp(-lam * (HI - LO)))) / lam

def sample_bkg(lam1, lam2, f, n):
    steep = np.random.uniform(size=n) < f
    return np.where(steep, sample_exp(lam1, n), sample_exp(lam2, n))

def make_toy(v, A_true):
    """One pseudo-experiment (m_pos, m_neg) from the parameters v of the nominal fit and asymmetry A_true."""
    out = []
    for side, sign in (("pos", -1), ("neg", +1)):
        n_s = np.random.poisson(v["N_s"] * (1 + sign * A_true) / 2)       # Poisson yields
        n_b = np.random.poisson(v[f"N_b_{side}"])
        m = np.concatenate([sample_gauss(v["mu"], v["sigma"], n_s),
                            sample_bkg(v[f"lam1_{side}"], v[f"lam2_{side}"], v[f"f_{side}"], n_b)])
        out.append(m)
    return out

A_GRID = [-0.10, -0.07, A_hat, -0.02, 0.0, 0.05]
N_TOY = {a: (60 if a == A_hat else 30) for a in A_GRID}       # report: 1000 at the fitted value, 300 at the others
t0 = time.time()
toy_rows = []
for a_true in A_GRID:
    for k in range(N_TOY[a_true]):
        mp, mn = make_toy(best_vals, a_true)
        A, sig, t, q0, ok = one_replica(mp, mn, a_true)
        toy_rows.append((a_true, A, sig, t, q0, ok))
toys = pd.DataFrame(toy_rows, columns=["A_true", "A", "sigma", "t", "q0", "valid"])
print(f"{len(toys)} toys, {(~toys.valid).sum()} not valid ({time.time() - t0:.0f} s)")
toys = toys[toys.valid]

In [ ]:
print(f"{'A_true':>8s} {'n':>4s} {'bias':>9s} {'spread/<Hesse>':>15s} {'pull mean':>10s} {'pull sd':>8s} "
      f"{'cov68':>6s} {'cov95':>6s} {'P(q0>3.84)':>11s} {'P(q0>9)':>8s}")
for a_true, g in toys.groupby("A_true"):
    pull = (g.A - a_true) / g.sigma
    print(f"{a_true:+8.4f} {len(g):4d} {g.A.mean() - a_true:+9.5f} {g.A.std() / g.sigma.mean():15.2f} {pull.mean():+10.2f} "
          f"{pull.std():8.2f} {np.mean(g.t <= 1):6.2f} {np.mean(g.t <= 3.84):6.2f} {np.mean(g.q0 > 3.84):11.2f} {np.mean(g.q0 > 9):8.2f}")
print("report (1000 toys at the fitted value): bias <= 0.0008, pulls width 0.94-1.05, cov68 0.696 +- 0.015, "
      "cov95 0.954 +- 0.007, size 0.051 +- 0.013, power(-0.02) 0.68")

The size is the line $A_{\rm true}=0$ of the last columns (P($q_0>3.84$) ≈ 0.05), the power the other lines.
With $\sim2\times10^4$ candidates an asymmetry of $-0.02$ is rejected only about two times out of three: below
$\sim0.02$ the test is not sensitive. The $5.2\sigma$ of the data is an *extrapolation* of the $\chi^2_1$ tail, which
the toys only probe down to $\sim3\sigma$.

**Neyman belt.** For each $A_{\rm true}$ of the grid the central 68 % and 95 % ranges of $\hat A$ (percentiles of the
toys); the interval of $A_{\rm true}$ whose belt contains the observed $\hat A$ is the confidence interval.

In [ ]:
belt = toys.groupby("A_true").A.quantile([0.025, 0.16, 0.5, 0.84, 0.975]).unstack()
fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(belt.index, belt[0.025], belt[0.975], color=C["freq"], alpha=0.2, label="95 %")
ax.fill_between(belt.index, belt[0.16], belt[0.84], color=C["freq"], alpha=0.4, label="68 %")
ax.plot(belt.index, belt.index, "k:", lw=1)
ax.axhline(A_hat, color=C["model"], label="observed $\\hat A$")
ax.set(xlabel="$A_{true}$", ylabel="$\\hat A$"); ax.legend()
plt.tight_layout(); plt.show()
# interval of A_true whose 68 % belt contains the observed value (linear interpolation of the belt edges)
lo_edge = np.interp(A_hat, belt[0.84].values, belt.index.values)    # A_true where the upper edge crosses A_hat
hi_edge = np.interp(A_hat, belt[0.16].values, belt.index.values)    # A_true where the lower edge crosses A_hat
print(f"Neyman 68% interval: [{lo_edge:+.4f}, {hi_edge:+.4f}]   (report: [-0.0528, -0.0351]; Minos [-0.0526, -0.0356])")

**Dependence on the momentum of the $B$.** If the asymmetry were produced by a detector effect that depends on the
kinematics it would vary with $p_T$ or $p$. We split the sample in terciles and refit each (report, cut appendix:
$A_{\rm raw}$ stable within uncertainties).

In [ ]:
for var in ("pT", "p"):
    edges_q = sample[var].quantile([0, 1/3, 2/3, 1]).values
    for k in range(3):
        s = sample[(sample[var] >= edges_q[k]) & (sample[var] <= edges_q[k + 1])]
        f = fit_joint(s.m[s.q > 0].to_numpy(), s.m[s.q < 0].to_numpy(), best_vals); f.hesse()
        print(f"{var:3s} tercile {k + 1} [{edges_q[k]/1e3:5.1f}, {edges_q[k + 1]/1e3:5.1f}] GeV  n={len(s)}  "
              f"A = {f.values['A']:+.4f} +- {f.errors['A']:.4f}")

## 7 · Systematic uncertainty

The nominal result depends on choices that the data do not fix. We vary them **one at a time** (13 variants in four
groups), refit and look at the shift $\Delta=A^{\rm var}-A^{\rm nom}$:

| group | variants |
|---|---|
| window | lower edge 5050, 5150 MeV; upper edge 5450, 5700 MeV |
| background | single exponential; shape shared by the two charges; charge-symmetric background ($A_b=0$) |
| signal | Crystal Ball instead of Gaussian; $\mu,\sigma$ separate for each charge |
| preselection | `ProbK` > 0.4, 0.6; `ProbPi` < 0.4, 0.6 |

To vary the model without rewriting it 13 times, `fit_variant` builds the two charge models from a few options.
With the default options it is exactly the nominal model (checked below).

In [ ]:
from scipy.stats import crystalball

def fit_variant(s, lo=LO, hi=HI, n_exp=2, shared_shape=False, sym_bkg=False, signal="gauss", per_charge=False):
    """Joint fit of the sample s (columns m, q) with the model options listed above. Returns (Minuit, models)."""
    mp, mn = s.m[s.q > 0].to_numpy(), s.m[s.q < 0].to_numpy()
    N = len(s)
    models, start = {}, {"N_s": 0.78 * N, "A": -0.04}
    limits = {"A": (-1, 1), "N_s": (0, 2 * N)}
    for side, sign in (("pos", -1), ("neg", +1)):
        # parameter names of this charge (a name used by both charges is shared by iminuit)
        sig_names = ["mu", "sigma"] if not per_charge else [f"mu_{side}", f"sigma_{side}"]
        if signal == "cb":
            sig_names += ["beta", "m_cb"]
        sfx = "" if shared_shape else f"_{side}"
        bkg_names = [f"lam1{sfx}"] + ([f"lam2{sfx}", f"f{sfx}"] if n_exp == 2 else [])
        yld = "N_b" if sym_bkg else f"N_b_{side}"
        names = ["N_s", "A"] + sig_names + [yld] + bkg_names

        def model(m, *p, names=names, sign=sign, sig_names=sig_names, bkg_names=bkg_names, yld=yld):
            v = dict(zip(names, p))
            n_sig = v["N_s"] * (1 + sign * v["A"]) / 2
            n_bkg = v[yld] / 2 if sym_bkg else v[yld]             # symmetric background: N_b / 2 per charge
            if signal == "gauss":
                sig = gauss_pdf(m, v[sig_names[0]], v[sig_names[1]], lo, hi)
            else:
                mu, sg = v[sig_names[0]], v[sig_names[1]]
                z = crystalball.cdf((hi - mu) / sg, v["beta"], v["m_cb"]) - crystalball.cdf((lo - mu) / sg, v["beta"], v["m_cb"])
                sig = crystalball.pdf((m - mu) / sg, v["beta"], v["m_cb"]) / sg / z
            if n_exp == 2:
                bkg = bkg_pdf(m, v[bkg_names[0]], v[bkg_names[1]], v[bkg_names[2]], lo, hi)
            else:
                bkg = exp_pdf(m, v[bkg_names[0]], lo, hi)
            return n_sig + n_bkg, n_sig * sig + n_bkg * bkg

        models[side] = (model, names)
        for n in names:                                           # starting values and limits by parameter name
            if n.startswith("mu"): start[n] = 5284.0; limits[n] = (5260, 5310)
            elif n.startswith("sigma"): start[n] = 18.0; limits[n] = (3, 40)
            elif n == "beta": start[n] = 1.8; limits[n] = (0.3, 5)
            elif n == "m_cb": start[n] = 10.0; limits[n] = (1.01, 50)
            elif n.startswith("N_b"): start[n] = 0.22 * N * (1 if sym_bkg else 0.5); limits[n] = (0, 2 * N)
            elif n.startswith("lam1"): start[n] = 0.03 if n_exp == 2 else 0.004; limits[n] = (1e-5, 0.1)
            elif n.startswith("lam2"): start[n] = 0.002; limits[n] = (1e-5, 0.1)
            elif n.startswith("f"): start[n] = 0.15; limits[n] = (0, 1)
    nll = (cost.ExtendedUnbinnedNLL(mp, models["pos"][0], name=models["pos"][1])
           + cost.ExtendedUnbinnedNLL(mn, models["neg"][0], name=models["neg"][1]))
    f = Minuit(nll, **start)
    for n, lim in limits.items():
        f.limits[n] = lim
    f.migrad(); f.hesse()
    return f, models

def binned_chi2(f, models, s, lo=LO, hi=HI, width=10.0):
    """Pearson chi2 in bins of 10 MeV, both charges, for a fit made by fit_variant."""
    edges = np.arange(lo, hi + 1e-9, width)
    chi2_ = 0.0
    for side in ("pos", "neg"):
        model, names = models[side]
        nu = expected_counts(model, [f.values[n] for n in names], edges)
        mm = s.m[s.q > 0] if side == "pos" else s.m[s.q < 0]
        obs = np.histogram(mm, edges)[0]
        chi2_ += np.sum((obs - nu)**2 / nu)
    return chi2_

# check: default options = nominal model
f_nom, models_nom = fit_variant(sample)
print(f"fit_variant(default): A = {f_nom.values['A']:+.5f}   nominal fit: A = {fit.values['A']:+.5f}")
assert abs(f_nom.values["A"] - fit.values["A"]) < 1e-4

**Combination rule.**
1. A variant that *changes the sample* (window, preselection) gives a nested sample: part of the shift is a
   statistical fluctuation of expected size $\sigma_B=\sqrt{|\sigma_{\rm var}^2-\sigma_{\rm nom}^2|}$ (Barlow
   criterion) [EXTRA]. It contributes only if $|\Delta|>\sigma_B$.
2. A variant rejected by the data ($\Delta\chi^2>25$ on the same sample, the single exponential) is excluded.
3. In each group the largest remaining $|\Delta|$ is the group uncertainty; groups are independent choices and are
   added in quadrature.

In [ ]:
def select(lo=LO, hi=HI, k=0.5, pi=0.5):
    s = d[(d.m > lo) & (d.m < hi) & (d.min_probk > k) & (d.max_probpi < pi)]
    return s.reset_index(drop=True)

VARIANTS = {   # name: (group, selection options, model options)
    "window_lo5050":       ("window", dict(lo=5050.0), dict(lo=5050.0)),
    "window_lo5150":       ("window", dict(lo=5150.0), dict(lo=5150.0)),
    "window_hi5450":       ("window", dict(hi=5450.0), dict(hi=5450.0)),
    "window_hi5700":       ("window", dict(hi=5700.0), dict(hi=5700.0)),
    "bkg_single_exp":      ("background", {}, dict(n_exp=1)),
    "bkg_shared_shape":    ("background", {}, dict(shared_shape=True)),
    "bkg_symmetric":       ("background", {}, dict(sym_bkg=True)),
    "signal_crystal_ball": ("signal", {}, dict(signal="cb")),
    "signal_per_charge":   ("signal", {}, dict(per_charge=True)),
    "presel_probk_0.4":    ("preselection", dict(k=0.4), {}),
    "presel_probk_0.6":    ("preselection", dict(k=0.6), {}),
    "presel_probpi_0.4":   ("preselection", dict(pi=0.4), {}),
    "presel_probpi_0.6":   ("preselection", dict(pi=0.6), {}),
}
SAMPLE_CHANGING = ("window", "preselection")
REPORT_A = {"window_lo5050": -0.0445, "window_lo5150": -0.0446, "window_hi5450": -0.0442, "window_hi5700": -0.0455,
            "bkg_single_exp": -0.0481, "bkg_shared_shape": -0.0481, "bkg_symmetric": -0.0472,
            "signal_crystal_ball": -0.0447, "signal_per_charge": -0.0442,
            "presel_probk_0.4": -0.0473, "presel_probk_0.6": -0.0384, "presel_probpi_0.4": -0.0421, "presel_probpi_0.6": -0.0448}

A0, s0 = f_nom.values["A"], f_nom.errors["A"]
chi2_nom = binned_chi2(f_nom, models_nom, sample)
syst_rows = []
for name, (group, sel_opt, mod_opt) in VARIANTS.items():
    s = select(**sel_opt)
    f, models = fit_variant(s, **mod_opt)
    shift, sig = f.values["A"] - A0, f.errors["A"]
    changes = group in SAMPLE_CHANGING
    sigma_B = np.sqrt(abs(sig**2 - s0**2)) if changes else np.nan
    rejected = (not changes) and (binned_chi2(f, models, s) - chi2_nom > 25)
    counts = (not rejected) and ((not changes) or abs(shift) > sigma_B)
    syst_rows.append(dict(variant=name, group=group, n=len(s), A=f.values["A"], shift=shift, sigma_B=sigma_B,
                          rejected=rejected, counts=counts, report_A=REPORT_A[name]))
syst = pd.DataFrame(syst_rows)
print(syst.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

group_unc = syst[syst.counts].groupby("group").shift.apply(lambda x: x.abs().max())
group_unc = group_unc.reindex(["window", "background", "signal", "preselection"]).fillna(0.0)
syst_total = np.sqrt((group_unc**2).sum())
print("\nuncertainty by group:", group_unc.round(4).to_dict())
print(f"total systematic = {syst_total:.4f}   (report: 0.0014, 0.0040, 0.0006, 0.0058 -> 0.0072)")
print(f"\nResult:  A_raw = {A0:+.4f} +- {s0:.4f} (stat) +- {syst_total:.4f} (syst)     report: -0.0441 +- 0.0085 +- 0.0072")

The **preselection** dominates through one single variant, `ProbK > 0.6`. Section 8 asks whether it is a trend
or a fluctuation.

## 8 · Stability scans

### 8.1 Fine scan of the PID cuts, one at a time

Each sample is nested in (or contains) the nominal one, so the shift is compared with $\sigma_B$. A trend with the
cut would indicate a real dependence (e.g. misidentified decays entering when the cut is loosened); a scatter
around zero within $\sim1\sigma_B$ a fluctuation. The report scans in steps of 0.05, as here.

In [ ]:
scan = []
for var, grid_ in (("k", np.arange(0.30, 0.851, 0.05)), ("pi", np.arange(0.15, 0.701, 0.05))):
    for cut in grid_:
        if abs(cut - 0.5) < 1e-9: continue
        s = select(**{var: float(cut)})
        f, _ = fit_variant(s)
        sig_B = np.sqrt(abs(f.errors["A"]**2 - s0**2))
        scan.append(dict(var=var, cut=cut, n=len(s), shift=f.values["A"] - A0, sigma_B=sig_B))
scan = pd.DataFrame(scan)

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (var, lab, nominal_cut) in zip(ax, (("k", "ProbK > cut", 0.5), ("pi", "ProbPi < cut", 0.5))):
    g = scan[scan["var"] == var]
    a.errorbar(g.cut, g["shift"], g.sigma_B, fmt="o", color=C["freq"], capsize=3)
    a.axhline(0, color="k", lw=0.8); a.axvline(nominal_cut, color="gray", ls="-.")
    a.set(xlabel=lab, ylabel="$A^{var}-A^{nom}$")
plt.tight_layout(); plt.show()
g = scan[(scan["var"] == "k") & (scan.cut > 0.5)]
print("tight ProbK: shifts / sigma_B =", (g["shift"] / g.sigma_B).round(2).tolist(), "  (report: scatter within 1.3 sigma_B, no trend)")
g = scan[(scan["var"] == "k") & (scan.cut < 0.5)]
print("loose ProbK: shifts           =", g["shift"].round(4).tolist(), "  (report: monotonic)")

### 8.2 Stability against the fit window (grid of the two edges)

In [ ]:
lefts, rights = [5050.0, 5100.0, 5150.0, 5200.0], [5450.0, 5500.0, 5600.0, 5700.0]
grid_A = np.zeros((len(lefts), len(rights)))
for i, lo_ in enumerate(lefts):
    for j, hi_ in enumerate(rights):
        f, _ = fit_variant(select(lo=lo_, hi=hi_), lo=lo_, hi=hi_)
        grid_A[i, j] = f.values["A"]
print("A_raw over the grid of the window edges (rows: lower edge, columns: upper edge)")
print(pd.DataFrame(grid_A, index=lefts, columns=rights).round(4).to_string())
print(f"max |shift| = {np.abs(grid_A - A0).max():.4f}   (report, wider grid: 0.0048)")

## 9 · Local asymmetry in the Dalitz plane

Is the asymmetry the same everywhere in phase space? Each candidate is placed in the **Dalitz plane** with the two
squared masses of the opposite-sign pairs, $m^2(H_1H_2)$ and $m^2(H_1H_3)$. Since $H_2,H_3$ are interchangeable
we sort them: $m^2_{\rm lo}=\min$, $m^2_{\rm hi}=\max$.

* Fixed grid $8\times8$ on $[0,16]\times[0,30]$ GeV$^2$, defined **before** looking at the asymmetry.
* In every bin with at least 20 candidates: $A_i=(N_i^--N_i^+)/N_i$, $\sigma(A_i)=\sqrt{(1-A_i^2)/N_i}$ and the
  significance with respect to the global asymmetry, $S_i=(A_i-A_{\rm glob})/\sigma(A_i)$.
* **Homogeneity test**: $\chi^2=\sum_iS_i^2$ with (number of bins $-1$) degrees of freedom.

This is a *counting* analysis: no background subtraction.

In [ ]:
lo2 = (np.minimum(sample.m12, sample.m13) / 1e3)**2        # GeV^2
hi2 = (np.maximum(sample.m12, sample.m13) / 1e3)**2
qv = sample.q.to_numpy()
el, eh = np.linspace(0, 16, 9), np.linspace(0, 30, 9)

def dalitz_map(lo2, hi2, q, min_events=20):
    """Counts and local asymmetry in the 8x8 grid; returns n, A_i, S_i, A_glob."""
    n_pos = np.histogram2d(lo2[q > 0], hi2[q > 0], [el, eh])[0]
    n_neg = np.histogram2d(lo2[q < 0], hi2[q < 0], [el, eh])[0]
    n = n_pos + n_neg
    a_glob = (np.sum(q < 0) - np.sum(q > 0)) / len(q)
    with np.errstate(divide="ignore", invalid="ignore"):
        a_i = (n_neg - n_pos) / n
        s_i = (a_i - a_glob) / np.sqrt((1 - a_i**2) / n)
    ok = n >= min_events
    return n, np.where(ok, a_i, np.nan), np.where(ok, s_i, np.nan), a_glob

n_i, a_i, s_i, a_glob = dalitz_map(lo2.to_numpy(), hi2.to_numpy(), qv)
ok = ~np.isnan(s_i)
chi2_h, ndof_h = np.nansum(s_i**2), ok.sum() - 1
print(f"global A = {a_glob:+.4f};  populated bins: {ok.sum()};  bins with |S|>3: {np.sum(np.abs(s_i[ok]) > 3)}")
print(f"homogeneity: chi2 = {chi2_h:.1f} for {ndof_h} d.o.f., p = {chi2.sf(chi2_h, ndof_h):.1e}     (report: 80.8/31, p = 3e-6)")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a, (img, lab, kw) in zip(ax, ((np.where(n_i > 0, n_i, np.nan), "candidates per bin", dict(cmap="Greys")),
                                  (a_i, "$A_i$", dict(cmap="coolwarm", vmin=-0.5, vmax=0.5)),
                                  (s_i, "$S_i$", dict(cmap="coolwarm", vmin=-4, vmax=4)))):
    im = a.imshow(img.T, origin="lower", aspect="auto", extent=[0, 16, 0, 30], **kw)
    plt.colorbar(im, ax=a, label=lab); a.set(xlabel="$m^2_{lo}$ [GeV$^2$]", ylabel="$m^2_{hi}$ [GeV$^2$]"); a.grid(False)
plt.tight_layout(); plt.show()

**Null test by charge permutation** [EXTRA]. If we shuffle the charge labels among the candidates, the Dalitz
distribution and the global asymmetry are unchanged but any *local* asymmetry is destroyed: $\chi^2$ must then follow
its distribution under uniformity. The observed value must lie far in the tail. (Report: 2000 permutations.)

In [ ]:
chi2_perm = []
for _ in range(500):
    q_shuffled = np.random.permutation(qv)
    _, _, s_p, _ = dalitz_map(lo2.to_numpy(), hi2.to_numpy(), q_shuffled)
    chi2_perm.append(np.nansum(s_p**2))
chi2_perm = np.array(chi2_perm)
print(f"permutations: mean {chi2_perm.mean():.1f} +- {chi2_perm.std():.1f}, max {chi2_perm.max():.1f};  "
      f"observed {chi2_h:.1f}, fraction >= observed: {np.mean(chi2_perm >= chi2_h):.4f}   (report: 31.5 +- 8.4, none above 80.8)")

**Where is the asymmetry?** The $\phi(1020)$ appears as a band at the kinematic edge ($m^2_{\rm lo}\approx1.04$ GeV$^2$)
and the charm resonances ($D^0$, $J/\psi$, $\chi_{c0}$) as bands. We count the asymmetry in the $\phi$ band and in
the bins that are significant. These numbers describe *a posteriori* a region selected because it is significant:
an illustration, not a new significance.

In [ ]:
def count_region(mask):
    q_ = qv[mask]; a = (np.sum(q_ < 0) - np.sum(q_ > 0)) / len(q_)
    return len(q_), a, np.sqrt((1 - a**2) / len(q_))

m_lo_mev = np.minimum(sample.m12, sample.m13).to_numpy()
phi = np.abs(m_lo_mev - 1019.461) < 10                                   # phi(1020) band
bins_sig = (lo2 < 4) & (hi2 > 7.5) & (hi2 < 11.25)                        # the two bins with |S| > 3
bins_sig = bins_sig.to_numpy()
for lab, mask in (("all candidates", np.ones(len(qv), bool)), ("phi(1020) band", phi),
                  ("significant bins", bins_sig), ("significant bins without phi", bins_sig & ~phi)):
    n_, a_, da_ = count_region(mask)
    print(f"{lab:30s} n = {n_:6d}   A_count = {a_:+.4f} +- {da_:.4f}")
print("report: phi band -0.004 +- 0.023;  significant bins without phi -0.145 +- 0.019")

**Magnet polarity.** The left–right detection asymmetry changes sign with the magnet polarity, while a CP asymmetry
does not. The counting asymmetry of the significant region in each polarity:

In [ ]:
for pol in ("down", "up"):
    mask = bins_sig & (sample.magnet == pol).to_numpy()
    n_, a_, da_ = count_region(mask)
    print(f"magnet {pol:4s}: n = {n_}, A_count = {a_:+.3f} +- {da_:.3f}")
print("report: -0.129 +- 0.023 (down), -0.126 +- 0.028 (up)")

### Region of the published analysis

The LHCb publication on the same data measured the asymmetry in the region $1.2<m^2_{\rm lo}<2.0$ GeV$^2$,
$m^2_{\rm hi}<15$ GeV$^2$ and found $A_{\rm raw}=-0.239\pm0.020$. The region is fixed by the paper, not chosen on
our data, so the comparison is *a priori*. We fit the nominal model to the candidates in the region.

In [ ]:
in_paper = ((lo2 > 1.2) & (lo2 < 2.0) & (hi2 < 15)).to_numpy()
reg = sample[in_paper]
f_reg, _ = fit_variant(reg)
print(f"candidates in the region: {len(reg)}  (report: 2146)")
print(f"A_raw (region) = {f_reg.values['A']:+.3f} +- {f_reg.errors['A']:.3f}     (report, frequentist: -0.237 +- 0.025; "
      f"published: -0.239 +- 0.020)")

### Charm vetoes

$B^\pm\to D^0K^\pm$, $J/\psi K^\pm$, $\chi_{c0}K^\pm$ with the resonance decaying to $K^+K^-$ are not charmless. A
candidate is rejected if one of its two opposite-sign pairs falls within $\pm30$ MeV ($D^0$, $J/\psi$) or $\pm60$ MeV
($\chi_{c0}$) of the mass of the resonance. The vetoes are a robustness check, not part of the systematic.

In [ ]:
CHARM = {"D0": (1864.84, 30.0), "Jpsi": (3096.9, 30.0), "chic0": (3414.7, 60.0)}
def veto(s, states):
    keep = np.ones(len(s), bool)
    for st in states:
        m0, hw = CHARM[st]
        keep &= ~((np.abs(s.m12 - m0) < hw) | (np.abs(s.m13 - m0) < hw))
    return s[keep]

for lab, states in (("D0 veto", ["D0"]), ("D0, J/psi, chic0 vetoes", ["D0", "Jpsi", "chic0"])):
    s = veto(sample, states)
    f, _ = fit_variant(s)
    print(f"{lab:26s} n = {len(s):6d}   A_raw = {f.values['A']:+.4f} +- {f.errors['A']:.4f}")
print("report: n = 18744, -0.0493 +- 0.0087  and  n = 16367, -0.0421 +- 0.0094")

## 10 · Summary

What was done, in order, and what it says:

1. **Counting** gives a first number; the **fit** separates signal and background: $A_{\rm raw}=-0.044\pm0.0085$.
2. The **profile likelihood** and Wilks' theorem give the intervals and reject $A=0$ at about $5\sigma$.
3. The **bootstrap** and the **toys** show that the estimator is unbiased, that the intervals cover, and how small
   an asymmetry the test could resolve.
4. The **13 variants** give the systematic uncertainty, dominated by the PID requirement; the fine scan shows that
   the dominant variant is a fluctuation.
5. The asymmetry is **not uniform** in the Dalitz plane: it sits outside the $\phi(1020)$, in the region of the
   published analysis.

The same likelihood is analysed with a Bayesian method in notebook 02.

In [ ]:
print(f"Total running time of this notebook: {(time.time() - T0) / 60:.1f} min")